In [ ]:
from __future__ import annotations

import os
import shutil
from pathlib import Path

import numpy as np

if Path.cwd().name == "notebooks":
    os.chdir("..")

import logging
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from config.config import METRICS_DIR, OUTPUTS_DIR, RUN_DIR
from config.runs_vars import RUNS_METADATA

plt.rcParams["figure.dpi"] = 144

In [ ]:
def save_plot(fig: plt.Figure, save_dir: Path, filename: str) -> None:
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_dir / f"{filename}.png", bbox_inches="tight")
    plt.close(fig)


def wrap_to_latex(text: str) -> str:
    if not (text.startswith("$") and text.endswith("$")):
        text = f"${text}$"
    return text


def plot_detaild_analysis(
    metrics_df: pd.DataFrame,
    metric: str,
    run_col_name: str,
    experiment_col: str,
    left_col: str,
    right_col: str,
    left_col_name: str,
    right_col_name: str,
    run_separator: str = "__",
    precision: int = 2,
    subfolder_name: str | None = None,
    filename_suffix: str = "",
    save_plots: bool = False,
    figsize: tuple[int, int] = (13, 8),
) -> None:
    metrics_df[run_col_name] = metrics_df[left_col].str.cat(metrics_df[right_col], sep=run_separator)
    metrics_df[left_col_name] = metrics_df[run_col_name].str.split(run_separator).str[0]
    metrics_df[right_col_name] = metrics_df[run_col_name].str.split(run_separator).str[1]

    if subfolder_name is None:
        subfolder_name = "plots"
    date_str = datetime.today().strftime("%Y-%m-%d")
    save_dir = METRICS_DIR / date_str / subfolder_name

    plot_defs = [
        (
            f"boxplot-by-{run_col_name.lower()}",
            lambda ax: plot_box(
                metrics_df, run_col_name, metric, palette="Set2", ax=ax, rotation=90, consistent_prefix_color=True
            ),
        ),
        (
            f"boxplot-by-{experiment_col.lower()}",
            lambda ax: plot_box(metrics_df, experiment_col, metric, palette="Set3", ax=ax),
        ),
        (
            f"{metric}--heatmap-{experiment_col.lower()}",
            lambda ax: plot_heatmap(metrics_df, run_col_name, experiment_col, metric, precision, ax=ax),
        ),
        ("histogram-metric-distribution", lambda ax: plot_histogram(metrics_df, metric, ax=ax)),
        (
            f"{metric}--heatmap-{left_col_name.lower()}-vs-{right_col_name.lower()}",
            lambda ax: plot_heatmap(metrics_df, left_col_name, right_col_name, metric, precision, ax=ax),
        ),
        (
            f"{metric}--heatmap-{run_col_name.lower()}-by-{left_col_name.lower()}-and-{right_col_name.lower()}",
            lambda ax: plot_combined_heatmap(
                metrics_df, run_col_name, left_col_name, right_col_name, metric, precision, ax=ax
            ),
        ),
        (
            f"boxplot-by-{left_col_name.lower()}",
            lambda ax: plot_box(metrics_df, left_col_name, metric, palette="viridis", ax=ax, rotation=0),
        ),
        (
            f"boxplot-by-{right_col_name.lower()}",
            lambda ax: plot_box(metrics_df, right_col_name, metric, palette="coolwarm", ax=ax, rotation=0),
        ),
    ]

    if save_plots:
        for idx, (name, plot_fn) in enumerate(plot_defs):
            fig, ax = plt.subplots(figsize=figsize)
            plot_fn(ax)
            ax.set_title(name.replace("-", " ").title())
            plt.tight_layout()
            save_plot(fig, save_dir, f"{idx}_{name}{filename_suffix}")
        return

    fig, axes = plt.subplots(4, 2, figsize=(28, 34))
    fig.suptitle(f"{metric} Metric Analysis Across Models, Experiments, and Experiment Splits", fontsize=16)
    axes = axes.flatten()
    for ax, (_, plot_fn) in zip(axes, plot_defs):
        plot_fn(ax)
    plt.tight_layout(rect=[0, 0, 1, 0.96])


def plot_box(
    data: pd.DataFrame,
    x_col: str,
    y_col: str,
    palette: str,
    ax: plt.Axes,
    rotation: int = 34,
    consistent_prefix_color: bool = False,
) -> None:
    if consistent_prefix_color:
        data = data.copy()
        unique_prefixes = data[x_col].str.split("-").str[0].unique()

        palette_colors = sns.color_palette(palette, n_colors=len(unique_prefixes))
        color_dict = dict(zip(unique_prefixes, palette_colors))

        box_plot = sns.boxplot(ax=ax, data=data, x=x_col, y=y_col, color="white", fliersize=0, linewidth=1.5)
        for patch, label in zip(box_plot.patches, data[x_col].unique()):
            patch.set_facecolor(color_dict[label.split("-")[0]])
    else:
        sns.boxplot(ax=ax, data=data, x=x_col, y=y_col, palette=palette, hue=x_col, legend=False)

    sns.stripplot(ax=ax, data=data, x=x_col, y=y_col, color="black", alpha=0.5, jitter=True, size=4)

    ax.set_xlabel(wrap_to_latex(x_col))
    ax.set_ylabel(y_col)
    ax.set_xticks(ax.get_xticks())
    ax.set_xticklabels([wrap_to_latex(t.get_text()) for t in ax.get_xticklabels()], rotation=rotation)


def plot_heatmap(data: pd.DataFrame, row_col: str, col_col: str, val_col: str, precision: int, ax: plt.Axes) -> None:
    pivot = data.pivot_table(index=row_col, columns=col_col, values=val_col, sort=True)
    pivot = pivot.reindex(index=data[row_col].unique(), columns=data[col_col].unique())
    sns.heatmap(ax=ax, data=pivot, cmap="coolwarm", annot=True, fmt=f".{precision}f", linewidths=0.5)
    ax.set_xlabel(wrap_to_latex(col_col))
    ax.set_ylabel(wrap_to_latex(row_col))
    ax.set_xticks(ax.get_xticks())
    ax.set_xticklabels([wrap_to_latex(t.get_text()) for t in ax.get_xticklabels()], rotation=34)
    ax.set_yticklabels([wrap_to_latex(t.get_text()) for t in ax.get_yticklabels()])


def plot_combined_heatmap(
    data: pd.DataFrame, row_col: str, col1: str, col2: str, val_col: str, precision: int, ax: plt.Axes
) -> None:
    p1 = data.pivot_table(index=row_col, columns=col1, values=val_col)
    p2 = data.pivot_table(index=row_col, columns=col2, values=val_col)
    combined = pd.concat([p1, p2], axis=1)
    sns.heatmap(ax=ax, data=combined, cmap="coolwarm", annot=True, fmt=f".{precision}f", linewidths=0.5)
    ax.set_xlabel(f"{col1} and {col2}")
    ax.set_ylabel(wrap_to_latex(row_col))
    ax.set_xticks(ax.get_xticks())
    ax.set_xticklabels([wrap_to_latex(t.get_text()) for t in ax.get_xticklabels()], rotation=34)
    ax.set_yticklabels([wrap_to_latex(t.get_text()) for t in ax.get_yticklabels()])


def plot_histogram(data: pd.DataFrame, metric: str, ax: plt.Axes) -> None:
    sns.histplot(ax=ax, data=data[metric], bins=20, kde=True, color="purple")
    ax.set_xlabel(metric)
    ax.set_ylabel("Frequency")


In [ ]:
PROMPT_NAMES_MAP = {
    "direct_entity_ontological": "P",
    "sequential_hierarchy_ontological": "P_{EC}",
    "direct_entity": "P^{NLF}",
    "sequential_hierarchy": "P_{EC}^{NLF}",
    "direct_entity_with_synonyms": "P_{S}^{NLF}",
    "sequential_hierarchy_with_synonyms": "P_{EC+S}^{NLF}",
    # "all_data_dummy": "P_{A}^{D}",
    # "only_names": "P_{N}^{D}",
    # "only_with_children": "P_{C}^{D}",
    # "only_with_parents": "P_{P}^{D}",
    # "with_hierarchy": "P_{H}^{D}",
}

MODEL_NAMES_MAP = {
    "gemini-1.5-flash": "Flash.1.5",
    "gemini-2.0-flash": "Flash.2.0",
    "gemini-2.0-flash-lite": "Flash.2.0.lite",
    "gemini-2.5-flash-preview-04-17": "Flash.2.5",
    "gpt-4o-mini": "gpt.4o.mini",
}

TASK_NAMES_MAP = {
    x: x
    for x in [
        "human-mouse",
        "ncit-doid",
        "omim-ordo",
        "snomed-fma.body",
        "snomed-ncit.neoplas",
        "snomed-ncit.pharm",
        "fma-nci",
        "fma-snomed",
        "snomed-nci",
    ]
}


def preprocess_metrics_df(metrics_df: pd.DataFrame) -> pd.DataFrame:
    metrics_df["Youden's index"] = metrics_df["Specificity"] + metrics_df["Sensitivity"] - 1
    metrics_df = metrics_df.rename(columns={"SubSet": "Matching Task"})

    if "Prompt" not in metrics_df.columns:
        metrics_df["Prompt"] = None

    if "Experiment" in metrics_df.columns:
        metrics_df.loc[metrics_df["Prompt"].isna(), "Prompt"] = metrics_df.loc[
            metrics_df["Prompt"].isna(), "Experiment"
        ]

    for col, mapping in [
        ("Matching Task", TASK_NAMES_MAP),
        ("Prompt", PROMPT_NAMES_MAP),
        ("Model", MODEL_NAMES_MAP),
    ]:
        metrics_df[col] = metrics_df[col].map(mapping)
        metrics_df[col] = pd.Categorical(metrics_df[col], categories=list(mapping.values()), ordered=True)

    metrics_df = metrics_df.dropna()
    return metrics_df.sort_values(["Model", "Matching Task", "Prompt"])

## Detailed plotting of the experiment results

In [ ]:
suffix = ""
metrics_df = preprocess_metrics_df(pd.read_csv(RUN_DIR / f"all_runs_metrics{suffix}.csv"))

metrics_df.head(7)
metrics_df["Prompt"].unique()

In [ ]:
experiment_col, left_col, right_col = "Matching Task", "Model", "Prompt"

for suffix in ["", "_reduced"]:
    for metric in ["Youden's index", "F1 Score"]:
        metrics_df = preprocess_metrics_df(pd.read_csv(RUN_DIR / f"all_runs_metrics{suffix}.csv"))
        plot_detaild_analysis(
            metrics_df=metrics_df,
            metric=metric,
            run_col_name="LLM-Prompt",
            experiment_col=experiment_col,
            left_col=left_col,
            right_col=right_col,
            left_col_name="Model",
            right_col_name="Prompt",
            run_separator="-",
            precision=3,
            subfolder_name=f"{metric}{suffix}",
            save_plots=False,
)
        plt.show()

## Copy best results

In [ ]:
prompt_type = "direct_entity_with_synonyms---"
model = "gemini-2.5-flash-preview-04-17"

In [ ]:
for run_metadata in RUNS_METADATA:
    run_name, model_name, subset_name, experiment_name = run_metadata
    run_dir = Path("runs") / run_name
    for file in run_dir.iterdir():
        if prompt_type in str(file) and model in str(file) and str(file).endswith(".csv"):
            logging.info(f"Processing {file} in {file}")
            new_file_path = OUTPUTS_DIR / "best_results" / datetime.now().strftime("%Y-%m-%d") / file.name
            new_file_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(file, new_file_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

outputs_dir = Path("./outputs")

datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    "gemini-2.0-flash-lite",
]

# few-shot experiments
experiment_type = "n_few_shots"
experiments = [
    f"{few_shot_version}/{n}"
    for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75)
    for few_shot_version in ["v0", "v1", "v2", "v3", "v4"]
]

records = []

for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for exp in experiments:
                run_subdir = f"{dataset}/{subset}/{model}/{experiment_type}/{exp}"
                path = outputs_dir / run_subdir / "results_reduced.csv"

                if not path.exists() or os.path.getsize(path) == 0:
                    # file missing
                    # records.append({
                    #     "Dataset": dataset,
                    #     "Subset": subset,
                    #     "Model": model,
                    #     "Experiment": exp,
                    #     "Prompt": "NO_FILE",
                    #     "Exists": False
                    # })
                    continue

                try:
                    df_local = pd.read_csv(path)
                except Exception as e:
                    print(f"Error reading {path}: {e}")
                    continue

                if "Prompt" not in df_local.columns:
                    print(f"Warning: Missing 'Prompt' column in {path}")
                    continue

                for prompt in df_local["Prompt"].unique():
                    records.append({
                        "Dataset": dataset,
                        "Subset": subset,
                        "Model": model,
                        "Experiment": exp,
                        "Prompt": prompt,
                        "Exists": True
                    })

df = pd.DataFrame(records)
df["TaskPrompt"] = df["Dataset"] + "/" + df["Subset"] + " | " + df["Prompt"]

print(f"Total records: {len(df)}")
print("Unique prompts:", df["Prompt"].unique())

sns.set_theme(style="whitegrid")

for model in df["Model"].unique():
    model_df = df[df["Model"] == model]

    pivot = model_df.pivot_table(
        index="TaskPrompt",
        columns="Experiment",
        values="Exists",
        aggfunc="first",
        fill_value=False
    )

    plt.figure(figsize=(16, max(6, 0.3 * len(pivot))))
    sns.heatmap(
        pivot,
        cmap=["#ffcccc", "#c6efce"],  # red = missing, green = exists
        cbar=False,
        linewidths=0.4,
        linecolor="gray"
    )

    plt.title(f"Missing Experiments for {model} (Grouped by Task & Prompt)", fontsize=14)
    plt.xlabel("Few-shot version / number")
    plt.ylabel("Task / Prompt")
    plt.xticks(rotation=90, ha="right")
    plt.tight_layout()
    plt.show()


## Gather outputs inference

In [ ]:
outputs_dir = Path("./outputs")
datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}
models = [
    "mistral.mistral-small-2402-v1:0",
    "mistral.mistral-large-2402-v1:0",
        "gemini-2.0-flash-lite",

    # "gemini-2.0-flash-lite",
    # "gemini-2.5-pro",
    # "gpt-5-nano",
    # "us.meta.llama3-2-3b-instruct-v1:0",
    # "anthropic.claude-instant-v1",
    # "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    "meta.llama3-70b-instruct-v1:0",
]
experiments_subtypes = {"syspromts": ["none", "basse", "natural_language", "ontology_aware", "synonym_aware"]}
experiments_subtypes = {"determinism": ["0", "1", "2", "3"]}
experiments_subtypes = {
    "n_few_shots": [
        f"{few_shot_version}/{n}"
        for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75)[:]
        for few_shot_version in ["v0", "v1", "v2", "v3", "v4"][:]
    ]
}
experiments_subtypes = {"open-weight-models": ["bedrock", "meta.llama3-70b-instruct-v1:0"]}
# experiments_subtypes = {"reasoning": ["no-reasoning", "with-reasoning"]}

dfs = []

for dataset, subsets_map in datasets_map.items():
    for set_name in subsets_map:
        for model in models:
            for experiment_type in experiments_subtypes:
                for experiment in experiments_subtypes[experiment_type]:
                    run_subdir = f"{dataset}/{set_name}/{model}/{experiment_type}/{experiment}"
                    path = outputs_dir / run_subdir / "results.csv"
                    if not path.exists():
                        print(f"Skipping missing {path}")
                        continue

                    if os.path.getsize(path) == 0:
                        print(f"Skipping empty {path}")
                        continue

                    try:
                        metrics_df = pd.read_csv(path)
                    except Exception as e:
                        print(f"Error reading {path}: {e}")
                        continue

                    # display(metrics_df)

                    for col, value in zip(
                        ["Dataset", "SubSet", "Model", "ExperimentType", "ExpName"],
                        [dataset, set_name, model, experiment_type, experiment],
                    ):
                        if col not in metrics_df.columns:
                            metrics_df[col] = value

                    metrics_df["Run"] = run_subdir
                    dfs.append(metrics_df)

results_df = pd.concat(dfs, ignore_index=True)
# results_df = preprocess_metrics_df(results_df)
# results_df.to_csv(METRICS_DIR / "open-source-models-2025-10-05.csv", index=False)

In [ ]:
outputs_dir = Path("./outputs")
datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}
models = [
    # "mistral.mistral-small-2402-v1:0",
    # "mistral.mistral-large-2402-v1:0",
    "gemini-2.0-flash-lite",
    # "gemini-2.5-pro",
    # "gpt-5-nano",
    # "us.meta.llama3-2-3b-instruct-v1:0",
    # "anthropic.claude-instant-v1",
    # "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    # "meta.llama3-70b-instruct-v1:0",
]
experiments_subtypes = {"syspromts": ["none", "basse", "natural_language", "ontology_aware", "synonym_aware"]}
experiments_subtypes = {"determinism": ["0", "1", "2", "3"]}
experiments_subtypes = {
    "n_few_shots": [
        f"{few_shot_version}/{n}"
        for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75)[:]
        for few_shot_version in ["v0", "v1", "v2", "v3", "v4"][:]
    ]
}
# experiments_subtypes = {"open-weight-models": ["bedrock", "meta.llama3-70b-instruct-v1:0"]}
# experiments_subtypes = {"reasoning": ["no-reasoning", "with-reasoning"]}

dfs = []

for dataset, subsets_map in datasets_map.items():
    for set_name in subsets_map:
        for model in models:
            for experiment_type in experiments_subtypes:
                for experiment in experiments_subtypes[experiment_type]:
                    run_subdir = f"{dataset}/{set_name}/{model}/{experiment_type}/{experiment}"
                    path = outputs_dir / run_subdir / "results_reduced.csv"
                    if not path.exists():
                        print(f"Skipping missing {path}")
                        continue

                    # check if file is not empty
                    if os.path.getsize(path) == 0:
                        print(f"Skipping empty {path}")
                        continue

                    try:
                        metrics_df = pd.read_csv(path)
                    except Exception as e:
                        print(f"Error reading {path}: {e}")
                        continue

                    # display(metrics_df)

                    for col, value in zip(
                        ["Dataset", "SubSet", "Model", "ExperimentType", "ExpName"],
                        [dataset, set_name, model, experiment_type, experiment],
                    ):
                        if col not in metrics_df.columns:
                            metrics_df[col] = value

                    metrics_df["Run"] = run_subdir
                    dfs.append(metrics_df)

results_df = pd.concat(dfs, ignore_index=True)
# results_df = preprocess_metrics_df(results_df)
# results_df.to_csv(METRICS_DIR / "few-shots-bedrock.csv", index=False)

In [ ]:
def plot_nfew_shots_analysis(sub_results: pd.DataFrame) -> None:
    sub_results["version"] = sub_results["ExpName"].str.split("/").str[0]
    sub_results["n_few_shots"] = sub_results["ExpName"].str.split("/").str[1].astype(int)

    for subset in sub_results["SubSet"].unique():
        prompts = sub_results["Prompt"].unique()

        n_prompts = len(prompts)
        ncols = 2
        nrows = int(np.ceil(n_prompts / ncols))

        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
        fig.suptitle(f"Subset: {subset}", fontsize=16)

        for i, prompt in enumerate(prompts):
            ax = axes[i // ncols, i % ncols]

            subset_df = sub_results[(sub_results["SubSet"] == subset) & (sub_results["Prompt"] == prompt)]
            if subset_df.empty:
                ax.set_visible(False)
                continue

            for version in subset_df["version"].unique():
                version_df = subset_df[subset_df["version"] == version]
                ax.plot(version_df["n_few_shots"], version_df["Youden's index"], marker="o", label=version)

            ax.set_title(f"Prompt: {prompt}")
            ax.set_xlabel("Number of few shots")
            ax.set_ylabel("Youden's index")
            ax.legend(title="Version")
            ax.grid(True)
            ax.set_ylim(0, 1)

        plt.tight_layout(rect=[0, 0, 1, 0.97])  # leave space for main title
        plt.show()


def plot_nfew_shots_analysis_mean(sub_results: pd.DataFrame) -> None:
    sub_results["version"] = sub_results["ExpName"].str.split("/").str[0]
    sub_results["n_few_shots"] = sub_results["ExpName"].str.split("/").str[1].astype(int)

    for subset in sub_results["SubSet"].unique():
        prompts = sub_results["Prompt"].unique()

        n_prompts = len(prompts)
        ncols = 2
        nrows = int(np.ceil(n_prompts / ncols))

        fig, axes = plt.subplots(
            nrows=nrows, ncols=ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False
        )
        fig.suptitle(f"Subset: {subset}", fontsize=16)

        for i, prompt in enumerate(prompts):
            ax = axes[i // ncols, i % ncols]

            subset_df = sub_results[
                (sub_results["SubSet"] == subset) & (sub_results["Prompt"] == prompt)
            ]
            if subset_df.empty:
                ax.set_visible(False)
                continue

            # group only by n_few_shots (mean/std across versions)
            grouped = (
                subset_df.groupby("n_few_shots")["Youden's index"]
                .agg(["mean", "std"])
                .reset_index()
            )

            ax.errorbar(
                grouped["n_few_shots"],
                grouped["mean"],
                yerr=grouped["std"],
                marker="o",
                capsize=4,
                label="Mean across versions",
            )

            ax.set_title(f"Prompt: {prompt}")
            ax.set_xlabel("Number of few shots")
            ax.set_ylabel("Youden's index (mean ± std)")
            ax.legend()
            ax.grid(True)
            ax.set_ylim(0, 1)

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import product

outputs_dir = Path("./outputs")

datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    # "mistral.mistral-large-2402-v1:0",
    "gemini-2.0-flash-lite",
]

experiment_type = "n_few_shots"
few_shot_versions = ["v0", "v1", "v2", "v3", "v4"]
few_shot_numbers = [0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75]

records = []

for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for n, v in product(few_shot_numbers, few_shot_versions):
                run_dir = outputs_dir / dataset / subset / model / experiment_type / f"{v}/{n}"
                if not run_dir.exists():
                    continue

                for csv_file in run_dir.glob("*.csv"):
                    if csv_file.name == "results_reduced.csv":
                        continue

                    prompt_name = csv_file.stem.replace("_results_reduced", "")
                    prompt_name = prompt_name.split(f"{model}_")[-1]

                    try:
                        df = pd.read_csv(csv_file)
                    except Exception as e:
                        print(f"Error reading {csv_file}: {e}")
                        continue

                    if "Prediction" not in df.columns:
                        print(f"Missing Prediction column in {csv_file}")
                        continue

                    counts = df["Prediction"].value_counts().to_dict()
                    total = len(df)

                    for pred_label, count in counts.items():
                        records.append(
                            {
                                "Dataset": dataset,
                                "Subset": subset,
                                "Task": f"{dataset}/{subset}",
                                "Model": model,
                                "FewShot": f"{v}/{n}",
                                "Prompt": prompt_name,
                                "Prediction": str(pred_label),
                                "Count": count,
                                "Total": total,
                            }
                        )

results_df = pd.DataFrame(records)
if results_df.empty:
    raise ValueError("No CSV data found — check paths and filenames!")

results_df["Proportion"] = results_df["Count"] / results_df["Total"]

print(f"Loaded {len(results_df)} prediction records.")
print("Example:")
print(results_df.head())

sns.set(style="whitegrid")

for (task, prompt), group_df in results_df.groupby(["Task", "Prompt"]):
    plt.figure(figsize=(14, 4))
    # group_df = group_df[group_df["Prediction"].isin(["error", "no_match", "match"])]
    sns.barplot(
        data=group_df,
        x="FewShot",
        y="Proportion",
        hue="Prediction",
        palette="Set2",
    )
    plt.title(f"{task} — {prompt}")
    plt.ylabel("Proportion of Predictions")
    plt.xlabel("Few-shot version / number")
    plt.xticks(rotation=90, ha="right")
    plt.legend(title="Prediction")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import product

outputs_dir = Path("./outputs")

datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    "gemini-2.0-flash-lite",
    # "mistral.mistral-large-2402-v1:0",
]

experiment_type = "n_few_shots"
few_shot_versions = ["v0", "v1", "v2", "v3", "v4"]
few_shot_numbers = [0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75]

records = []

# === COLLECT DATA ===
for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for v, n in product(few_shot_versions, few_shot_numbers):
                run_dir = outputs_dir / dataset / subset / model / experiment_type / f"{v}/{n}"
                if not run_dir.exists():
                    continue

                for csv_file in run_dir.glob("*.csv"):
                    if csv_file.name == "results_reduced.csv":
                        continue

                    prompt_name = csv_file.stem.replace("_results_reduced", "")
                    if f"{model}_" in prompt_name:
                        prompt_name = prompt_name.split(f"{model}_")[-1]

                    try:
                        df = pd.read_csv(csv_file)
                    except Exception as e:
                        print(f"Error reading {csv_file}: {e}")
                        continue

                    if "Prediction" not in df.columns:
                        continue

                    total = len(df)
                    if total == 0:
                        continue

                    false_count = (df["Prediction"].astype(str).str.lower() == "error").sum()
                    error_rate = false_count / total

                    if error_rate == 0:
                        continue

                    records.append({
                        "Task": f"{dataset}/{subset}",
                        "Model": model,
                        "Prompt": prompt_name,
                        "FewShot": f"{v}/{n}",
                        "ErrorRate": error_rate,
                    })

results_df = pd.DataFrame(records)
if results_df.empty:
    raise ValueError("No valid few-shot experiments with errors found!")

results_df["Label"] = results_df["Task"] + "/" + results_df["Prompt"] + "/" + results_df["FewShot"]

print(f"Loaded {len(results_df)} few-shot experiments with non-zero errors.")
print(results_df.head())

sns.set(style="whitegrid")

plt.figure(figsize=(max(14, len(results_df) * 0.25), 7))
sns.barplot(
    data=results_df.sort_values("ErrorRate", ascending=False),
    x="Label",
    y="ErrorRate",
    hue="Model",
    palette="Reds",
)

plt.title("Error Rate per Task / Prompt / Few-Shot", fontsize=15)
plt.ylabel("Error Rate (Error / Total)")
plt.xlabel("Task / Prompt / Few-Shot")
plt.xticks(rotation=90, ha="center", fontsize=8)
plt.legend(title="Model")
plt.tight_layout()
# plt.yscale("log")
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
plot_nfew_shots_analysis_mean(results_df[results_df["Model"] == "gemini-2.5-pro"])

In [ ]:
plot_nfew_shots_analysis(results_df[results_df["Model"] == "gpt-5-nano"])

In [ ]:
plot_nfew_shots_analysis(results_df[results_df["Model"] == "gemini-2.0-flash-lite"])

In [ ]:
plot_nfew_shots_analysis_mean(results_df[results_df["Model"] == "gemini-2.0-flash-lite"])

In [ ]:
results_df["version"] = results_df["ExpName"].str.split("/").str[0]
results_df["n_few_shots"] = results_df["ExpName"].str.split("/").str[1].astype(int)

sub_results = results_df
for subset in sub_results["SubSet"].unique():
    prompts = sub_results["Prompt"].unique()

    n_prompts = len(prompts)
    ncols = 2
    nrows = int(np.ceil(n_prompts / ncols))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
    fig.suptitle(f"Subset: {subset}", fontsize=16)

    for i, prompt in enumerate(prompts):
        ax = axes[i // ncols, i % ncols]

        subset_df = sub_results[(sub_results["SubSet"] == subset) & (sub_results["Prompt"] == prompt)]
        if subset_df.empty:
            ax.set_visible(False)
            continue

        for version in subset_df["version"].unique():
            version_df = subset_df[subset_df["version"] == version]
            ax.plot(version_df["n_few_shots"], version_df["Youden's index"], marker="o", label=version)

        ax.set_title(f"Prompt: {prompt}")
        ax.set_xlabel("Number of few shots")
        ax.set_ylabel("Youden's index")
        ax.legend(title="Version")
        ax.grid(True)
        ax.set_ylim(0, 1)

    plt.tight_layout(rect=[0, 0, 1, 0.97])  # leave space for main title
    plt.show()


In [ ]:
outputs_dir = Path("./outputs")
datasets_map = {
    "anatomy": ["human-mouse"],
    # "largebio": ["fma-nci"],
    # "bioml-2024": ["omim-ordo"],
}
# models = ["gemini-2.0-flash-lite"]  # , "gemini-2.0-flash"]
models = [
    "mistral.mistral-small-2402-v1:0",
    # "us.meta.llama3-2-3b-instruct-v1:0",
    # "anthropic.claude-instant-v1",
    # "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    "meta.llama3-70b-instruct-v1:0",
]
experiments_subtypes = {"syspromts": ["none", "basse", "natural_language", "ontology_aware", "synonym_aware"]}
experiments_subtypes = {"determinism": ["0", "1", "2", "3"]}
experiments_subtypes = {"n_few_shots": [str(n) for n in [0, 1, 2, 3, 4, 6, 8, 10, 12, 14, 17, 20, 25, 30, 35, 40, 50]]}
# experiments_subtypes = {"open-weight-models": ["bedrock", "meta.llama3-70b-instruct-v1:0"]}
# experiments_subtypes = {"reasoning": ["no-reasoning", "with-reasoning"]}

dfs = []

for dataset, subsets_map in datasets_map.items():
    for set_name in subsets_map:
        for model in models:
            for experiment_type in experiments_subtypes:
                for experiment in experiments_subtypes[experiment_type]:
                    run_subdir = f"{dataset}/{set_name}/{model}/{experiment_type}/{experiment}"
                    path = outputs_dir / run_subdir / "results.csv"
                    if not path.exists():
                        print(f"Skipping missing {path}")
                        continue

                    metrics_df = pd.read_csv(path)

                    # display(metrics_df)

                    for col, value in zip(
                        ["Dataset", "SubSet", "Model", "ExperimentType", "ExpName"],
                        [dataset, set_name, model, experiment_type, experiment],
                    ):
                        if col not in metrics_df.columns:
                            metrics_df[col] = value

                    metrics_df["Run"] = run_subdir
                    dfs.append(metrics_df)

results_df = pd.concat(dfs, ignore_index=True)
# results_df = preprocess_metrics_df(results_df)
results_df

In [ ]:
results_df.to_csv(METRICS_DIR/"tmp.csv", index=False)

In [ ]:
results_df

In [ ]:
results_df

In [ ]:
results_df[(results_df["Model"]=="Flash.2.0")].groupby(["Prompt", "ExpName"]).agg(
    {
        "Youden's index": ["mean"],
        "F1 Score": ["mean"],
        "Precision": ["mean"],
        "Recall": ["mean"],
    }
).round(3)

In [ ]:
metrics_df = results_df.copy()
metric = "Youden's index"

experiment_col, left_col, right_col = "Matching Task", "Prompt", "ExpName"

for metric in ["Youden's index", "F1 Score"]:
    plot_detaild_analysis(
        metrics_df=metrics_df,
        metric=metric,
        run_col_name="LLM-Prompt",
        experiment_col=experiment_col,
        left_col=left_col,
        right_col=right_col,
        left_col_name=left_col,
        right_col_name=right_col,
        run_separator="-",
        precision=4,
        subfolder_name=f"{next(iter(experiments_subtypes.keys()))}/{metric}",
        save_plots=False,
    )
    plt.show()

## Radar Charts

In [ ]:
def make_radar_chart(
    ax: plt.Axes,
    categories: list[str],
    values_list: list[list[float]],
    labels: list[str],
    colors: list[str],
    linestyles: list[str],
    r_range: tuple[float, float],
    title: str,
    scale: float = 1.01,
) -> None:
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)

    ax.set_rlabel_position(0)
    ax.set_ylim(*r_range)
    yticks = np.linspace(r_range[0], r_range[1], 4)
    ax.set_yticks(yticks)
    ax.set_yticklabels([f"{y:.3f}" for y in yticks], fontsize=9)

    for values, label, color, linestyle in zip(values_list, labels, colors, linestyles):
        values += values[:1]
        ax.set_ylim(ax.get_ylim()[0], np.max(values) * scale)
        ax.plot(angles, values, label=label, color=color, linestyle=linestyle, linewidth=1)
        ax.fill(angles, values, color=color, alpha=0.05)
        ax.grid(color="gray", linestyle="--", linewidth=0.5, alpha=0.5)

    ax.set_title(title, size=12, y=1.1)
    # plt.tight_layout(rect=[0, 0.01, 1, 0.93])

In [ ]:
f1_oracle_df = pd.read_csv(METRICS_DIR / "OASystem_with_oracle_f1.csv", sep='\t')
f1_oracle_df = f1_oracle_df[
    [
        "Task",
        '$OASystem(auto)$',
        # '$OASystem+Or^{GF2.0}$',
        '$OASystem+Or^{0}$',
        # '$OASystem+Or^{10}$',
        '$OASystem+Or^{20}$',
        '$OASystem+Or^{30}$',
        '$OASystem+Or^{LLM}$',
    ]
]
# renale OASystem to LogMap in all columns except Task
f1_oracle_df = f1_oracle_df.rename(columns={"$OASystem+Or^{LLM}$": "$LogMap+Or^{LLM}_{GF2.5}$"})
f1_oracle_df = f1_oracle_df.rename(columns=lambda x: x.replace("OASystem", "LogMap"))
labels = f1_oracle_df.columns[1:]
f1_oracle_df

In [ ]:
oracle_df = f1_oracle_df.copy()

categories1= oracle_df["Task"].tolist()
values_list1 = oracle_df.iloc[:, 1:].to_numpy().T.tolist()

colors = ["blue", "green", "purple", "red", "black"]
linestyles = ["solid", "dashed", "dotted", "dashdot", (0, (5, 2))]


plt.rcParams["figure.dpi"] =444

fig, axs = plt.subplots(1, 1, subplot_kw={"polar": True}, figsize=(8, 8))
make_radar_chart(axs, categories1, values_list1, labels, colors, linestyles, (0.5, 0.99), "", scale=1.03)

handles, _ = axs.get_legend_handles_labels()
fig.legend(handles, labels,loc="lower center", ncol=3, fontsize=10)

# fig.suptitle("Comparison against Oracles with different error rates", fontsize=14)
# plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.tight_layout()
# plt.subplots_adjust(hspace=0.3)

save_path = METRICS_DIR / datetime.today().strftime("%Y-%m-%d") / "OASystem" / "OA_with_oracle_radar_chart_all.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
# fig.savefig(save_path, bbox_inches="tight")

In [ ]:
idx = 4
oracle_df = f1_oracle_df.copy()

categories1, categories2 = oracle_df["Task"].tolist()[:idx], oracle_df["Task"].tolist()[idx:]
values_list1 = oracle_df.iloc[:, 1:].to_numpy()[:idx].T.tolist()
values_list2 = oracle_df.iloc[:, 1:].to_numpy()[idx:].T.tolist()

colors = ["blue", "green", "purple", "red", "black"]
linestyles = ["solid", "dashed", "dotted", "dashdot", (0, (5, 2))]


plt.rcParams["figure.dpi"] = 444

fig, axs = plt.subplots(1, 2, subplot_kw={"polar": True}, figsize=(14, 7))
axs = axs.flatten()

make_radar_chart(axs[0], categories1, values_list1, labels, colors, linestyles, (0.7, 0.9), "", scale=1.03)
make_radar_chart(axs[1], categories2, values_list2, labels, colors, linestyles, (0.55, 0.72), "", scale=1.05)

handles, _ = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=10)

# fig.suptitle("Comparison against Oracles with different error rates", fontsize=14)
plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)

save_path = METRICS_DIR / datetime.today().strftime("%Y-%m-%d") / "OASystem" / "OA_with_oracle_radar_chart_verical.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(save_path, bbox_inches="tight")

In [ ]:
exp_oracle_df = pd.read_csv(METRICS_DIR / "OASystem_with_oracle_f1.csv", sep="\t")

for col in exp_oracle_df.columns[:0:-1]:
    exp_oracle_df[col] -= exp_oracle_df.iloc[:, 1]
exp_oracle_df

In [ ]:
plt.figure(figsize=(12, 6))

for col in exp_oracle_df.columns[1:]:
    if "LLM}$" in col:
        plt.plot(exp_oracle_df["Task"], exp_oracle_df[col], label=col, color="black", marker="o")
    else:
        plt.plot(exp_oracle_df["Task"], exp_oracle_df[col], label=col, linestyle="--", marker="o")

plt.xticks(rotation=20, ha="center")
plt.ylabel(r"$\Delta$Score")
plt.title("Model Comparison across Tasks")
plt.legend()
plt.grid(True)
plt.tight_layout()

save_path = METRICS_DIR / datetime.today().strftime("%Y-%m-%d") / "OASystem" / "OA_with_oracle_line_chart.png"
# save_path.parent.mkdir(parents=True, exist_ok=True)
# plt.savefig(save_path, bbox_inches="tight")